<a href="https://colab.research.google.com/github/Adithyan773/demo10/blob/main/CNN_TOMATO_LEAVES_WRK.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install opendatasets legacy-cgi

In [ ]:
import pandas as pd
import opendatasets as od

In [ ]:
od.download('https://www.kaggle.com/datasets/kaustubhb999/tomatoleaf/croissant/download')

Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds
Your Kaggle username: qwertyui
Your Kaggle Key: ··········
Dataset URL: https://www.kaggle.com/datasets/kaustubhb999/tomatoleaf


100%|██████████| 179M/179M [00:00<00:00, 204MB/s]


In [ ]:
import os

main_dir = '/content/tomatoleaf/tomato/train'
sub_dir = os.listdir(main_dir)
img_exe = ('.jpg','.jepg','.png')

sub_dir = os.listdir(main_dir)

img_exe = ('.jpg','.jpeg','.png')

for sub in sub_dir:
  path = os.path.join(main_dir,sub)
  image_count = 0

  for filename in os.listdir(path):
    if filename.lower().endswith(img_exe):
      image_count += 1
  print(f"image count {image_count} and in  folder {sub} ")


image count 1000 and in  folder Tomato___Spider_mites Two-spotted_spider_mite 
image count 1000 and in  folder Tomato___Leaf_Mold 
image count 1000 and in  folder Tomato___Bacterial_spot 
image count 1000 and in  folder Tomato___healthy 
image count 1000 and in  folder Tomato___Late_blight 
image count 1000 and in  folder Tomato___Tomato_Yellow_Leaf_Curl_Virus 
image count 1000 and in  folder Tomato___Septoria_leaf_spot 
image count 1000 and in  folder Tomato___Tomato_mosaic_virus 
image count 1000 and in  folder Tomato___Target_Spot 
image count 1000 and in  folder Tomato___Early_blight 


In [ ]:
import os
import cv2

x = []
y = []

main_dir = '/content/tomatoleaf/tomato/train'
sub_dir = os.listdir(main_dir)
img_exe = ('.jpg','.jepg','.png')

sub_dir = os.listdir(main_dir)

img_exe = ('.jpg','.jpeg','.png')

for sub in sub_dir:
  path = os.path.join(main_dir,sub)
  for filename in os.listdir(path):
    if filename.lower().endswith(img_exe):
      img_path = os.path.join(path,filename)
      img = cv2.imread(img_path)
      if img is None:
        continue
      img_rgb = cv2.cvtColor(img , cv2.COLOR_BGR2RGB)
      resize_img = cv2.resize(img_rgb , (224,224))
      x.append(resize_img)
      if sub == 'Tomato___Spider_mites Two-spotted_spider_mite':
        y.append(0)
      elif sub == 'Tomato___Leaf_Mold':
        y.append(1)
      elif sub == 'Tomato___Bacterial_spot':
        y.append(2)
      elif sub == 'Tomato___healthy':
        y.append(3)
      elif sub == 'Tomato___Late_blight':
        y.append(4)
      elif sub == 'Tomato___Tomato_Yellow_Leaf_Curl_Virus':
        y.append(5)
      elif sub == 'Tomato___Septoria_leaf_spot':
        y.append(6)
      elif sub == 'Tomato___Tomato_mosaic_virus':
        y.append(7)
      elif sub == 'Tomato___Target_Spot':
        y.append(8)
      elif sub ==  'Tomato___Early_blight':
        y.append(9)

In [ ]:
import numpy as np

x = np.array(x) / 255.0

y = np.array(y)

In [ ]:
from sklearn.model_selection import train_test_split

x_train , x_test , y_train , y_test = train_test_split(x , y , test_size = 0.2, stratify = y , random_state=42)

In [ ]:
import tensorflow as tf

from tensorflow.keras import layers , models
from tensorflow.keras.applications import VGG16 , ResNet50V2

def build_vgg16(input_shape = (224,224,3), num_classes = 10):
  base_model = VGG16(weights = 'imagenet',input_shape = input_shape)
  base_model.trainable = False  # Freeze pre_train weight

  model = models.Sequential([
      base_model,
      layers.Dense(256,activation = 'relu'),
      layers.Dropout(0.5),
      layers.Dense(num_classes, activation = 'softmax')
  ], name = "VGG16_Mask_Classifier")

  return model

vgg_model = build_vgg16()

vgg_model.summary()

Model: "VGG16_Mask_Classifier"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ vgg16 (Functional)              │ (None, 1000)           │   138,357,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 256)            │       256,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 10)             │         2,570 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 138,616,370 (528.78 MB)

 Trainable params: 258,826 (1011.04 KB)

 Non-trainable params: 138,357,544 (527.79 MB)

In [ ]:
vgg_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history = vgg_model.fit(x_train , y_train , validation_data=(x_test,y_test), epochs = 10 , batch_size=32)